# Regression Analysis
## Importing libraries and creating a NaN filtered dataset

In [ ]:

import enum

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pandas as pd

penguins = sns.load_dataset("penguins").dropna() # dopna removes rows with NaN data points

penguins.describe()


## Check models with variables of each non-categorical variable

In [ ]:

variables = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
variables2 = ["species", "sex"]
xLen = max(len(variables), len(variables2))
fig, axes = plt.subplots(2, xLen, figsize=(15,8))
name = "CDV"


sns.set()

for i, var in enumerate(variables):
    model = smf.ols("body_mass_g ~" + var, data=penguins)
    result = model.fit()
    print(f"{name} {var}'s R²: {result.rsquared}")
    sns.scatterplot(
        data=penguins,
        x=var,
        y="body_mass_g",
        ax=axes[0][i],
        hue="sex",
        palette = {
          "Male": "tab:blue",
          "Female": "tab:pink",
        },
        style="species",
        legend=False
    )

for i, var in enumerate(variables2):
    model = smf.ols("body_mass_g ~" + var, data=penguins)
    result = model.fit()
    print(f"{name} {var}'s R²: {result.rsquared}")
    sns.boxplot(
        data=penguins,
        x=var,
        y="body_mass_g",
        ax=axes[1][i],
        legend=False
    )

sns.scatterplot(
    data=penguins,
    x=variables[-1],
    y="body_mass_g",
    ax=axes[0][-1],
    hue="sex",
    palette = {
      "Male": "tab:blue",
      "Female": "tab:pink",
    },
    style="species",
    legend="full"
)
handles, labels = axes[0][-1].get_legend_handles_labels()
axes[0][-1].legend().remove()
fig.legend(handles, labels, bbox_to_anchor=(1.0,0.5), loc="center left", borderaxespad=0)
fig.tight_layout()
fig.savefig(name + ".jpg")

plt.show()


Flipper length seems to be the best variable for body mass so far, as the R² value is closest to 1.
It can also be seen that both bill length and depth varies widely based on the species of penguin, which might be the reason for their low R² values. One can also see in the plot that the reason flipper length has a comparatively high R² value is that longer flipper length correlates with species and sex with higher body mass, which could be the real variables.
Notably, the bill depth has an inverse correlation with body mass. Although this is because there's a positive correlation within 2 different groupings in the plots, but the adelie and chinstrap penguins seem to have deeper bills that gentoo pegnuins despite their lower body mass, affecting the correlation.
The estimated formula for body_mass_g ~ flipper_length_mm are:
$\hat{y} = 50.1533*x + -5872.0927$
In other words, $\hat{\beta_1} = 50.1533$ and $\hat{\beta_0} = -5872.0927$

## Filtering by species and sex

In [ ]:

rows = []
for (sp, sx), g in penguins.groupby(["species", "sex"]):
  fit = smf.ols("body_mass_g ~ flipper_length_mm", data=g).fit()
  rows.append({"species": sp, "sex": sx, "n": len(g), "r": g["flipper_length_mm"].corr(g["body_mass_g"]), "slope": fit.params["flipper_length_mm"], "R2": fit.rsquared})
print(pd.DataFrame(rows).round(2))


Here we can see that within the categories of sexes and species, the slopes are way lower than the 50.1533 estimate of our model. This confirms the theory that flipper_length is a useful variable mostly due to differences between species.

## More useful plots
### Linear plot
Despite this, we can see that a linear regression model fits the data well, and that we don't need to transform any variables.

In [ ]:

filename = "linearRegression"
fig, axes = plt.subplots(1,1, figsize=(5,5))
sns.regplot(x=penguins["flipper_length_mm"], y=penguins["body_mass_g"], ax=axes, line_kws={"color": "black"}, scatter_kws={"s": 2})
plt.plot()
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
print(model.fit().summary())


The variable flipper_length_mm being $\hat{ \beta_1 } = 50.1533$ means that for every millimeter of flipper length extra measured on a penguin, it's likely for the response (measured body mass) to be 50.1533g heavier. A penguin with longer flippers are likely to have a heavier body mass.
The estimated 95% confidence interval for $\hat{ \beta_1 }$ is CI=[47.123, 53.183]. With $\alpha=0.5$, it seems the variable does affect body_mass_g positively.
The low p value shows we can be quite certain of the linear connection.

### TA plot

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
sns.scatterplot(
                x=result.fittedvalues,
                y=result.resid,
                ax=axes,
                s=8,
                )
plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.show()


It can be seen here that there seems to be 2 clusters, and something that looks like a subtle curve.

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
sns.scatterplot(
                data=penguins,
                x=result.fittedvalues,
                y=result.resid,
                ax=axes,
                s=8,
                hue="sex",
                palette = {
                  "Male": "tab:blue",
                  "Female": "tab:pink",
                },
                style="species"
                )
plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0)
plt.show()


The clusters aren't a sign of a poor model by themselves, but they do indicate there might be more to this graph. Earlier, we saw how the model doesn't hold up when filtering for the categories of sex and species, which we can tell here is likely the cause for the strange clustering.

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
ax = sns.scatterplot(
  x=result.fittedvalues,
  y=result.resid,
  ax=axes,
  s=8,
)

sns.regplot(
  x=result.fittedvalues,
  y= result.resid,
  lowess=True,
  scatter=False,
  color="red",
  ax=ax
)

plt.ylabel("Residual")
plt.xlabel("Predicted value")
ax.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.show()


Here we can see there's a clear curve in the TA plot. We'll see if it disappears when including more variables later.

### QQ plot

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
sm.qqplot(result.resid, line="45", fit=True)
plt.ylabel("Quantiles in standardized residuals")
plt.xlabel("Quantiles in normal distribution")
plt.title("QQ plot")
plt.show()


The QQ-plot can be seen here to seem fine, indicating a normal distribution. However, due to the issues with the TA-plot the model is still under question. The QQ-plot only shows the overall shape of the distribution, ignoring where it sits along the fitted values. Where these observations sit along the fitted values which we can observe in the TA-plot is what let us figure out the trends which make the model problematic.

## Model with species and sex
Now that we've seen the variable of flipper length might be hiding the effects of other variables, we'll create a multiple variable regression model, and drop each variable separately from the model.

In [ ]:
def genModel(vars, data=penguins):
  return smf.ols("body_mass_g ~ " + " + ".join(vars), data=data)

allVars = variables + variables2
fit = genModel(allVars).fit()
print(fit.summary())


In [ ]:

rAdjSqrdChanges = {}
for i, var in enumerate(allVars):
    newVars = allVars.copy()
    newVars.remove(var)
    rAdjSqrdChange = fit.rsquared_adj - genModel(newVars).fit().rsquared_adj
    rAdjSqrdChanges[rAdjSqrdChange] = var
    print(f"R² change when removing {var}: {rAdjSqrdChange}")

print(f"\nThe biggest R² change is: {max(rAdjSqrdChanges.keys())} and is caused by the variable {rAdjSqrdChanges[max(rAdjSqrdChanges.keys())]}")


We can see here that species has the biggest unique contribution, and is therefore a contender for the strongest variable.

In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins)
result = model.fit()
ax = sns.scatterplot(
    data=penguins,
    x=result.fittedvalues,
    y=result.resid,
    ax=axes,
    s=8,
    hue="sex",
    palette = {
      "Male": "tab:blue",
      "Female": "tab:pink",
    },
    style="species"
)

sns.regplot(
  x=result.fittedvalues,
  y= result.resid,
  lowess=True,
  scatter=False,
  color="red",
  ax=ax
)

plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0)
plt.show()


In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
ax = sns.scatterplot(
    data=penguins,
    x=fit.fittedvalues,
    y=fit.resid,
    ax=axes,
    s=8,
    hue="sex",
    palette = {
      "Male": "tab:blue",
      "Female": "tab:pink",
    },
    style="species"
)

sns.regplot(
  x=fit.fittedvalues,
  y= fit.resid,
  lowess=True,
  scatter=False,
  color="red",
  ax=ax
)

plt.ylabel("Residual")
plt.xlabel("Predicted value")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("TA plot")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0)
plt.show()


The new model no longer underestimates males and overestimates females, and the slight curve has been reduced. This seems to be an improvement.


In [ ]:

fig, axes = plt.subplots(1,1, figsize=(5,5))
sm.qqplot(fit.resid, line="45", fit=True)
plt.ylabel("Quantiles in standardized residuals")
plt.xlabel("Quantiles in normal distribution")
plt.title("QQ plot")
plt.show()


The QQ plot still indicates a normative distribution, with little change from the previous model's plot.

# Sex and Species weight

Now to  our question:
Do males tend to be heavier than females?
And are gentoo penguins the heaviest, followed by chinstrap?

We've seen indications of this in several of the earlier plots, but particularly in our final model, we can see their individual contributions directly with minimal noise.

In [ ]:

print(f"Male factor across all species: {fit.params["sex[T.Male]"]}")
newVars = allVars.copy()
newVars.remove("species")
species = ["Gentoo", "Chinstrap", "Adelie"]
for i, sp in enumerate(species):
    data = penguins[penguins["species"] == sp]
    newFit = genModel(newVars, data=data).fit()
    print(f"{sp}'s male factor: {newFit.params["sex[T.Male]"]}")


Here we can see that with female as default, the male factor affects the weight positively, meaning that males tend to be heavier than females. This also holds true within each subspecies of penguin. This means our hypothesis held true.

In [ ]:

print(f"Gentoo factor: {fit.params["species[T.Gentoo]"]}")
print(f"Chinstrap factor: {fit.params["species[T.Chinstrap]"]}")


Our hypothesis that gentoo penguins are the heaviest was correct, however our hypothesis that Chinstrap penguins are the second heaviest was wrong. We can see that adelie penguins are the default, and that gentoo penguins affect the weight positively, and chinstrap penguins affect it negatively. This means that the order of weights is actually gentoo, adelie, chinstrap; not gentoo, chinstrap, adelie.